# Kaggle Training: Libri4Mix DExFormer
This notebook clones the `Audiosep` repository, generates a 4-speaker configuration on the fly, and launches training using your generated Libri4Mix dataset.

In [ ]:
# 1. Setup Environment
!pip install -q speechbrain soundfile
!git clone https://github.com/LiquidMemories72/Audiosep.git
%cd Audiosep

### Generate 4-Speaker Configuration
Since the 4-speaker config is not committed yet, we dynamically generate it by patching the 5-speaker config.

In [ ]:
import os

base_cfg = "configs/kaggle_libri5mix.yaml"
new_cfg = "configs/kaggle_libri4mix.yaml"

with open(base_cfg, "r") as f:
    config = f.read()

# Patch hyperparameters for 4-speaker separation
config = config.replace("num_spks: 5", "num_spks: 4")
config = config.replace("gradient_accumulation_steps: 16", "gradient_accumulation_steps: 12")
config = config.replace("max_epochs: 300", "max_epochs: 250")
config = config.replace("lr_warmup_steps: 8000", "lr_warmup_steps: 6000")
# Patch dataset names in the config string
config = config.replace("libri5mix", "libri4mix")

with open(new_cfg, "w") as f:
    f.write(config)

print(f"Successfully generated {new_cfg} for 4-speaker training!")

### Start Training
> **IMPORTANT**: Update `DATASET_ROOT` below with the actual Kaggle path of your dataset output from the previous notebook. Check the right-hand **Data** panel under **Input** for the exact folder name.

In [ ]:
# 2. Define your Kaggle Dataset Path
# Replace 'YOUR-DATASET-NAME' with the actual dataset folder name you added to Kaggle.
DATASET_ROOT = "/kaggle/input/YOUR-DATASET-NAME"
OUTPUT_DIR = f"{DATASET_ROOT}/Output"

DATA_FOLDER = f"{OUTPUT_DIR}/Libri4Mix/wav16k/max"
TRAIN_DATA = f"{OUTPUT_DIR}/Libri4Mix/metadata/libri4mix_train-clean-100.csv"
VALID_DATA = f"{OUTPUT_DIR}/Libri4Mix/metadata/libri4mix_dev-clean.csv"
TEST_DATA = f"{OUTPUT_DIR}/Libri4Mix/metadata/libri4mix_test-clean.csv"

# 3. Start Training!
!torchrun --standalone --nproc_per_node=2 scripts/train.py configs/kaggle_libri4mix.yaml \
    --data_folder {DATA_FOLDER} \
    --train_data {TRAIN_DATA} \
    --valid_data {VALID_DATA} \
    --test_data {TEST_DATA}